In [1]:
# ONLY RUN THIS IF YOU'RE IN GOOGLE COLAB
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Thesis/Pintu-Air/notebooks')

# Verify you're in the right place
!pwd
!ls -la

Mounted at /content/drive
/content/drive/MyDrive/Thesis/Pintu-Air/notebooks
total 93805
-rw------- 1 root root  6303711 Dec 19 19:01 01_Result_Data_Cleaning_Part1.csv
-rw------- 1 root root  1720705 Dec 19 19:01 02_Data_Preperation.ipynb
-rw------- 1 root root  2300416 Dec 19 19:01 03_All_Data.csv
-rw------- 1 root root  1984161 Dec 19 19:01 04_X_test_all.csv
-rw------- 1 root root 37612340 Dec 19 19:01 04_X_train_all.csv
-rw------- 1 root root    46509 Dec 19 19:01 04_y_test.csv
-rw------- 1 root root   883281 Dec 19 19:01 04_y_train.csv
-rw------- 1 root root  1205202 Dec 19 19:01 05_X_test_binary.csv
-rw------- 1 root root 22929893 Dec 19 19:01 05_X_train_binary.csv
-rw------- 1 root root   584912 Dec 19 19:01 06_X_test_significant.csv
-rw------- 1 root root 11132982 Dec 19 19:01 06_X_train_significant.csv
-rw------- 1 root root    47887 Dec 19 19:01 11_XGBoost_all_GridSearch_Results.csv
-rw------- 1 root root  1532395 Dec 19 19:01 11_XGBoost_all.ipynb
-rw------- 1 root root    4800

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import RobustScaler
from itertools import product
import os
import warnings
import time
warnings.filterwarnings('ignore')

In [3]:
# GPU Configuration
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# ADD THIS BLOCK:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except RuntimeError as e:
        print(f"GPU setup error: {e}")

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled


In [4]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [5]:
y_train = (pd.read_csv("04_y_train.csv", index_col='Tanggal')).values
y_test = (pd.read_csv("04_y_test.csv", index_col='Tanggal')).values

X_train_df = pd.read_csv("06_X_train_significant.csv", index_col='Tanggal')
X_test_df = pd.read_csv("06_X_test_significant.csv", index_col='Tanggal')

In [6]:
# Feature Processing
categorical_cols = [col for col in X_train_df.columns if 'cuaca' in col]
numeric_cols = [col for col in X_train_df.columns if 'air' in col]

X_num_train = X_train_df[numeric_cols].copy()
X_cat_train = X_train_df[categorical_cols].copy()

X_num_test = X_test_df[numeric_cols].copy()
X_cat_test = X_test_df[categorical_cols].copy()

In [7]:
X_num_train

,manggarai_air_lag1,manggarai_air_lag2,manggarai_air_lag3,manggarai_air_lag4,manggarai_air_lag5,manggarai_air_lag6,manggarai_air_lag7,manggarai_air_lag9,manggarai_air_lag12,manggarai_air_lag14,...,depok_air_lag19,depok_air_lag20,depok_air_lag23,depok_air_lag24,katulampa_air_lag3,katulampa_air_lag4,katulampa_air_lag5,katulampa_air_lag14,katulampa_air_lag19,katulampa_air_lag21
Tanggal,,,,,,,,,,,,,,,,,,,,,
2021-10-17 00:00:00,565.0,565.0,565.0,600.0,585.0,570.0,570.0,570.0,570.0,560.0,...,85.0,90.0,95.0,80.0,10.0,10.0,10.0,10.0,10.0,10.0
2021-10-17 01:00:00,590.0,565.0,565.0,565.0,600.0,585.0,570.0,570.0,570.0,565.0,...,85.0,85.0,95.0,95.0,10.0,10.0,10.0,10.0,10.0,10.0
2021-10-17 02:00:00,575.0,590.0,565.0,565.0,565.0,600.0,585.0,570.0,570.0,570.0,...,85.0,85.0,90.0,95.0,10.0,10.0,10.0,10.0,10.0,10.0
2021-10-17 03:00:00,570.0,575.0,590.0,565.0,565.0,565.0,600.0,570.0,570.0,570.0,...,80.0,85.0,90.0,90.0,20.0,10.0,10.0,10.0,10.0,10.0
2021-10-17 04:00:00,565.0,570.0,575.0,590.0,565.0,565.0,565.0,585.0,570.0,570.0,...,80.0,80.0,85.0,90.0,20.0,20.0,10.0,10.0,10.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-01 07:00:00,670.0,670.0,670.0,670.0,670.0,670.0,670.0,670.0,680.0,680.0,...,115.0,115.0,140.0,150.0,30.0,30.0,30.0,30.0,30.0,30.0
2025-09-01 08:00:00,670.0,670.0,670.0,670.0,670.0,670.0,670.0,670.0,680.0,680.0,...,110.0,115.0,130.0,140.0,30.0,30.0,30.0,70.0,20.0,30.0
2025-09-01 09:00:00,680.0,670.0,670.0,670.0,670.0,670.0,670.0,670.0,680.0,680.0,...,110.0,110.0,130.0,130.0,30.0,30.0,30.0,70.0,20.0,30.0


In [8]:
X_cat_train

,manggarai_cuaca_lag1_hujan,manggarai_cuaca_lag2_hujan,manggarai_cuaca_lag4_hujan,manggarai_cuaca_lag5_hujan,manggarai_cuaca_lag7_hujan,manggarai_cuaca_lag9_hujan,manggarai_cuaca_lag12_hujan,manggarai_cuaca_lag19_hujan,depok_cuaca_lag1_hujan,depok_cuaca_lag2_hujan,depok_cuaca_lag4_hujan,depok_cuaca_lag5_hujan,depok_cuaca_lag6_hujan,depok_cuaca_lag7_hujan,depok_cuaca_lag9_hujan,depok_cuaca_lag22_hujan,katulampa_cuaca_lag14_hujan,katulampa_cuaca_lag18_hujan
Tanggal,,,,,,,,,,,,,,,,,,
2021-10-17 00:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2021-10-17 01:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2021-10-17 02:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2021-10-17 03:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2021-10-17 04:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-01 07:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2025-09-01 08:00:00,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
2025-09-01 09:00:00,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [9]:
# Scaling
# scaler_X = MinMaxScaler()
# scaler_y = MinMaxScaler()
scaler_X = RobustScaler()
scaler_y = RobustScaler()

X_num_train_scaled = scaler_X.fit_transform(X_num_train)
X_num_test_scaled = scaler_X.transform(X_num_test)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()


print(f"\n{'='*60}\nData Train and Test Distribution:\n{'='*60}")

print(f'Jumlah data: {len(X_num_train) + len(X_num_test)}')

print(f'Jumlah data X train: {len(X_num_train)}')
print(f'Jumlah data X test: {len(X_num_test)}')

print(f'Jumlah data y train: {len(y_train)}')
print(f'Jumlah data y test: {len(y_test)}')


Data Train and Test Distribution:
Jumlah data: 35760
Jumlah data X train: 33972
Jumlah data X test: 1788
Jumlah data y train: 33972
Jumlah data y test: 1788


In [10]:
# Combine Features
X_train = np.concatenate([X_num_train_scaled, X_cat_train.values], axis=1)
X_test = np.concatenate([X_num_test_scaled, X_cat_test.values], axis=1)

print(f"Features: {X_train.shape[1]} total ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")

Features: 52 total (34 numeric, 18 categorical)


In [11]:
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []

    for i in range(seq_len, len(X)):
        # Take current timestep and previous (seq_len-1) steps SINCE X IS ALREADY IN LAGGED FORMAT
        X_seq.append(X[i - seq_len + 1 : i + 1])
        # Current time step as target
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)

In [12]:
def build_model(seq_len, n_features, units, dropout, lr):
    model = Sequential()
    for i, unit in enumerate(units):
        return_seq = i < len(units) - 1
        if i == 0:
            model.add(LSTM(unit, return_sequences=return_seq, input_shape=(seq_len, n_features)))
        else:
            model.add(LSTM(unit, return_sequences=return_seq))
        model.add(Dropout(dropout))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

In [13]:
def evaluate_model(model, X_seq, y_true, scaler_y):
    y_pred_scaled = model.predict(X_seq, verbose=0)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Calculate metrics
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred)
    }

In [14]:
def save_learning_curves(history, combo_id, params):
    os.makedirs('saved plots', exist_ok=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['loss'], label='Train')
    ax1.plot(history.history['val_loss'], label='Validation')
    ax1.set_title(f'Loss - Combo {combo_id}')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history.history['mae'], label='Train')
    ax2.plot(history.history['val_mae'], label='Validation')
    ax2.set_title(f'MAE - Combo {combo_id}')
    ax2.set_ylabel('MAE')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    # Handle list to string conversion for filename safely
    units_str = str(params['lstm_units']).replace('[','').replace(']','').replace(',','-')
    filename = f"LSTM_Combo{combo_id}_Seq{params['sequence_length']}_Unit{units_str}_Dr{params['dropout_rate']}.png"
    plt.savefig(f"saved plots/{filename}", dpi=100, bbox_inches='tight')
    plt.close()

In [23]:
def lstm_grid_search(X_train, y_train_scaled, X_test, y_test, scaler_y, param_grid):
    param_combinations = list(product(*param_grid.values()))
    param_names = list(param_grid.keys())
    total_combinations = len(param_combinations)

    print(f"Starting grid search: {total_combinations} combinations")
    results = []

    for i, params in enumerate(param_combinations):
        param_dict = dict(zip(param_names, params))
        print(f"\nCombination {i+1}/{total_combinations}: {param_dict}")

        try:
            # 1. Create Sequences
            X_seq, y_seq = create_sequences(X_train, y_train_scaled, param_dict['sequence_length'])

            # Validation Split (20%)
            val_size = int(len(X_seq) * 0.2)
            X_train_fold = X_seq[:-val_size]
            y_train_fold = y_seq[:-val_size]
            X_val_fold = X_seq[-val_size:]
            y_val_fold = y_seq[-val_size:]

            # 2. Build Model
            model = build_model(
                param_dict['sequence_length'],
                X_seq.shape[2],
                param_dict['lstm_units'],
                param_dict['dropout_rate'],
                param_dict['learning_rate']
            )

            callbacks = [
                EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, min_delta=0.0001),
                ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=0.00001)
            ]

            # 3. Train with Timer
            start_time = time.time() # Start Timer

            history = model.fit(
                X_train_fold, y_train_fold,
                epochs=param_dict['epochs'],
                batch_size=param_dict['batch_size'],
                validation_data=(X_val_fold, y_val_fold),
                callbacks=callbacks,
                verbose=0 # Silent training to keep output clean
            )

            training_time = time.time() - start_time # End Timer

            # Save Plot
            save_learning_curves(history, i+1, param_dict)

            # 4. Evaluate (Train & Test)
            # Train Set (Using the fold used for training to see fit)
            y_train_fold_orig = scaler_y.inverse_transform(y_train_fold.reshape(-1, 1)).flatten()
            train_metrics = evaluate_model(model, X_train_fold, y_train_fold_orig, scaler_y)

            # Test Set
            X_test_seq, _ = create_sequences(X_test, y_test.flatten(), param_dict['sequence_length'])
            y_test_actual = y_test[param_dict['sequence_length']:]
            test_metrics = evaluate_model(model, X_test_seq, y_test_actual, scaler_y)

            # 5. Format Result for CSV
            result_entry = {
                'Seq': param_dict['sequence_length'],
                'Unit': str(param_dict['lstm_units']), # Convert list to string for CSV
                'Dropout': param_dict['dropout_rate'],
                'LR': param_dict['learning_rate'],
                'Batch Size': param_dict['batch_size'],
                'Epoch': len(history.history['loss']), # Actual epochs run

                'training_time_sec': round(training_time, 8),

                'train_rmse': round(train_metrics['rmse'], 8),
                'train_mae': round(train_metrics['mae'], 8),
                'train_mape': round(train_metrics['mape'], 8),
                'train_r2': round(train_metrics['r2'], 8),

                'test_rmse': round(test_metrics['rmse'], 8),
                'test_mae': round(test_metrics['mae'], 8),
                'test_mape': round(test_metrics['mape'], 8),
                'test_r2': round(test_metrics['r2'], 8)
            }
            results.append(result_entry)
            df_results = pd.DataFrame(results)
            df_results.to_csv('33_LSTM_significant_Results.csv', index=False)

            print(f"   -> Time: {training_time:.2f}s | Train RMSE: {train_metrics['rmse']:.4f} | Test RMSE: {test_metrics['rmse']:.4f}")

        except Exception as e:
            print(f"Error in combination {i+1}: {str(e)}")
            # Append error row to keep track
            results.append({'Seq': param_dict['sequence_length'], 'test_rmse': 999999, 'error': str(e)})
            df_results = pd.DataFrame(results)
            df_results.to_csv('33_LSTM_significant_Results.csv', index=False)

    return pd.DataFrame(results)

In [24]:
param_grid = {
    'sequence_length': [1, 12, 24],
    'lstm_units': [
        [16,8],
        [128],
        [32,16],
        [64],
        [64,32]
    ],
    'dropout_rate': [0.4, 0.2],
    'learning_rate': [0.0001, 0.0005, 0.002],
    'batch_size': [32],
    'epochs': [40]
}

In [25]:
df_results = lstm_grid_search(X_train, y_train_scaled, X_test, y_test, scaler_y, param_grid)

Starting grid search: 90 combinations

Combination 1/90: {'sequence_length': 1, 'lstm_units': [16, 8], 'dropout_rate': 0.4, 'learning_rate': 0.0001, 'batch_size': 32, 'epochs': 40}
   -> Time: 248.27s | Train RMSE: 17.3230 | Test RMSE: 15.7785

Combination 2/90: {'sequence_length': 1, 'lstm_units': [16, 8], 'dropout_rate': 0.4, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 40}
   -> Time: 115.81s | Train RMSE: 20.3544 | Test RMSE: 15.4437

Combination 3/90: {'sequence_length': 1, 'lstm_units': [16, 8], 'dropout_rate': 0.4, 'learning_rate': 0.002, 'batch_size': 32, 'epochs': 40}
   -> Time: 51.30s | Train RMSE: 18.0309 | Test RMSE: 15.7968

Combination 4/90: {'sequence_length': 1, 'lstm_units': [16, 8], 'dropout_rate': 0.2, 'learning_rate': 0.0001, 'batch_size': 32, 'epochs': 40}
   -> Time: 245.09s | Train RMSE: 16.6968 | Test RMSE: 15.5869

Combination 5/90: {'sequence_length': 1, 'lstm_units': [16, 8], 'dropout_rate': 0.2, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 4

In [ ]:
time.sleep(5)  # give logs time to flush
os._exit(0)

In [ ]:
# Parameter Grid
param_grid = {
    'sequence_length': [24],
    'lstm_units': [[64, 32, 16], [128, 64, 32]],
    'dropout_rate': [0.2],
    'learning_rate': [0.001, 0.005, 0.01],
    'batch_size': [64],
    'epochs': [50]
}

In [ ]:
# Parameter Grid
param_grid = {
    'sequence_length': [36, 48, 72],
    'lstm_units': [[32], [64]],
    'dropout_rate': [0.3, 0.5],
    'learning_rate': [0.001],
    'batch_size': [64],
    'epochs': [30]
}

In [ ]:
param_grid = {
    'sequence_length': [36],  # Best performer
    'lstm_units': [[64]],     # Best architecture
    'dropout_rate': [0.4, 0.5, 0.6],  # Higher dropout to reduce overfitting
    'learning_rate': [0.0005, 0.001],  # Slightly lower LR
    'batch_size': [32],       # Smaller batch for better generalization
    'epochs': [25]            # Fewer epochs with early stopping
}

In [ ]:
param_grid = {
    # 1. SEQUENCE LENGTH (4 options)
    # Focus around the sweet spot but expand range
    'sequence_length': [24, 36, 48, 60],  # Add 60h for daily+weekly patterns

    # 2. ARCHITECTURE (3 options)
    # Focus on best performers only
    'lstm_units': [
        [32],      # Simpler for less overfitting
        [64],      # Current best performer
        [48]       # Sweet spot between 32 and 64
    ],

    # 3. DROPOUT STRATEGY (5 options)
    # CRITICAL: Address overfitting with higher dropout
    'dropout_rate': [0.4, 0.5, 0.6, 0.7, 0.8],  # Push dropout higher!

    # 4. LEARNING RATE (1 option - FIXED)
    # Fix at optimal value to reduce combinations
    'learning_rate': [0.0005],  # Sweet spot from your results

    # 5. BATCH SIZE (1 option - fixed)
    'batch_size': [32],  # Smaller batch for better generalization

    # 6. EPOCHS (1 option - fixed)
    'epochs': [40]  # Longer training with early stopping
}

In [ ]:
# Run Grid Search
results, best_params, best_score = lstm_grid_search(X_train, y_train_scaled, X_test, y_test, scaler_y, param_grid)

print(f"\nGrid search completed. Best RMSE: {best_score:.4f}")
print(f"Best parameters: {best_params}")

Starting grid search: 60 combinations

Combination 1/60: {'sequence_length': 24, 'lstm_units': [32], 'dropout_rate': 0.4, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 40}
Progress: 1.7%
Epoch 1/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - loss: 0.2751 - mae: 0.3322 - val_loss: 0.1859 - val_mae: 0.1487 - learning_rate: 5.0000e-04
Epoch 2/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 0.0979 - mae: 0.1731 - val_loss: 0.1746 - val_mae: 0.1194 - learning_rate: 5.0000e-04
Epoch 3/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - loss: 0.0850 - mae: 0.1539 - val_loss: 0.1720 - val_mae: 0.1110 - learning_rate: 5.0000e-04
Epoch 4/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 0.0798 - mae: 0.1433 - val_loss: 0.1707 - val_mae: 0.1143 - learning_rate: 5.0000e-04
Epoch 5/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 0.0777 - mae: 0.1360 - val_loss: 0.1695 - val_mae: 0.1045 - learning_rate: 5.0000e-04
Epoch 6/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 0.07

In [ ]:
results_df = pd.DataFrame([r for r in results if 'error' not in r])
results_df.to_csv('lstm_arobust_long_grid_search_results.csv', index=False)
print(f"Results saved to lstm_grid_search_results.csv")

Results saved to lstm_grid_search_results.csv


In [ ]:
# Create sequences with best parameters
SEQUENCE_LENGTH = best_params['sequence_length']
X_seq_final, y_seq_final = create_sequences(X_train, y_train_scaled, SEQUENCE_LENGTH)

In [ ]:
# Build final model with best parameters
final_model = build_model(
    best_params['sequence_length'],
    X_seq_final.shape[2],
    best_params['lstm_units'],
    best_params['dropout_rate'],
    best_params['learning_rate']
)

In [ ]:
# Train on ALL training data (no validation split for final model)
# callbacks = [
#     EarlyStopping(monitor='loss', patience=20, restore_best_weights=True, verbose=1),
#     ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, min_lr=0.00001, verbose=1)
# ]

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        min_delta=0.0001
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=0.00001
    )
]

In [ ]:
final_history = final_model.fit(
    X_seq_final, y_seq_final,
    epochs=best_params['epochs'] + 20,  # Allow more epochs for final training
    batch_size=best_params['batch_size'],
    callbacks=callbacks,
    verbose=1
)

Epoch 1/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - loss: 0.2073 - mae: 0.2709 - learning_rate: 5.0000e-04
Epoch 2/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - loss: 0.1152 - mae: 0.1663 - learning_rate: 5.0000e-04
Epoch 3/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - loss: 0.1071 - mae: 0.1513 - learning_rate: 5.0000e-04
Epoch 4/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - loss: 0.1010 - mae: 0.1423 - learning_rate: 5.0000e-04
Epoch 5/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 15s 17ms/step - loss: 0.0974 - mae: 0.1372 - learning_rate: 5.0000e-04
Epoch 6/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - loss: 0.0974 - mae: 0.1349 - learning_rate: 5.0000e-04
Epoch 7/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step - loss: 0.0954 - mae: 0.1311 - learning_rate: 5.0000e-04
Epoch 8/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - loss: 0.0922 - mae: 0.1267 - learning_rate: 5.0000e-04
Epoch 9/60
926/926 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - loss: 0.0924 - mae: 0.1269 - learning_rate: 5.0000e-04
E

In [ ]:
# Training evaluation (convert scaled targets back to original scale)
y_train_actual = scaler_y.inverse_transform(y_seq_final.reshape(-1, 1)).flatten()
final_train_metrics = evaluate_model(final_model, X_seq_final, y_train_actual, scaler_y)

# Test evaluation
X_test_seq, _ = create_sequences(X_test, y_test.flatten(), SEQUENCE_LENGTH)
y_test_actual = y_test[SEQUENCE_LENGTH:]
final_test_metrics = evaluate_model(final_model, X_test_seq, y_test_actual, scaler_y)

print(f"\nFinal Model Performance:")
print(f"TRAINING SET:")
print(f"  Train RMSE: {final_train_metrics['rmse']:.4f}")
print(f"  Train R2: {final_train_metrics['r2']:.4f}")
print(f"  Train MAE: {final_train_metrics['mae']:.4f}")
print(f"  Train MAPE: {final_train_metrics['mape']:.4f}")

print(f"\nTEST SET:")
print(f"  Test RMSE: {final_test_metrics['rmse']:.4f}")
print(f"  Test R2: {final_test_metrics['r2']:.4f}")
print(f"  Test MAE: {final_test_metrics['mae']:.4f}")
print(f"  Test MAPE: {final_test_metrics['mape']:.4f}")

print(f"\nOVERFITTING ANALYSIS:")
print(f"  RMSE Difference (Train - Test): {final_train_metrics['rmse'] - final_test_metrics['rmse']:.4f}")
print(f"  R2 Difference (Train - Test): {final_train_metrics['r2'] - final_test_metrics['r2']:.4f}")
print(f"  MAE Difference (Train - Test): {final_train_metrics['mae'] - final_test_metrics['mae']:.4f}")
print(f"  MAPE Difference (Train - Test): {final_train_metrics['mape'] - final_test_metrics['mape']:.4f}")

# Save final model
final_model.save('best_lstm_model.keras')
print("\nFinal model saved as 'best_lstm_model.keras'")


Final Model Performance:
TRAINING SET:
  Train RMSE: 20.6354
  Train R2: 0.7919
  Train MAE: 10.5227
  Train MAPE: 2.4391

TEST SET:
  Test RMSE: 25.9417
  Test R2: 0.1516
  Test MAE: 9.8446
  Test MAPE: 2.7461

OVERFITTING ANALYSIS:
  RMSE Difference (Train - Test): -5.3063
  R2 Difference (Train - Test): 0.6403
  MAE Difference (Train - Test): 0.6781
  MAPE Difference (Train - Test): -0.3070

Final model saved as 'best_lstm_model.keras'


: 